# Evaluate Performance Using Ridge

In [14]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor
import optuna
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge

In [2]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [3]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


In [4]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance


In [5]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

## **Data Preprocessing** 

In [6]:
preprocessor = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())

## **Feature Selection** 

In [10]:
# Conduct feature selection using shap_select.

def perform_feature_selection(X_train, y_train):
    results_dict = {}

    X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2)
    X_tr_fe = preprocessor.fit_transform(X_tr)
    X_te_fe = preprocessor.transform(X_te)
    X_te_df = pd.DataFrame(X_te_fe, columns=X_train.columns, index=y_te.index)
    
    
    for target in y_train.columns:
        model = XGBRegressor(n_estimators=1000, verbosity = 0, eval_metric='rmse', objective='reg:squarederror')
        model.fit(X_tr_fe, y_tr[target], eval_set=[(X_te_fe, y_te[target])])

        selected_df = shap_select(model, X_te_df, y_te[target], task="regression", threshold=0.05)
        results_dict[target] = selected_df

    return results_dict  

In [11]:
results = perform_feature_selection(X_train, y_train)

[0]	validation_0-rmse:61.14167
[1]	validation_0-rmse:52.73011
[2]	validation_0-rmse:47.30802
[3]	validation_0-rmse:42.90090
[4]	validation_0-rmse:40.19608
[5]	validation_0-rmse:38.74684
[6]	validation_0-rmse:37.21890
[7]	validation_0-rmse:36.75916
[8]	validation_0-rmse:36.00083
[9]	validation_0-rmse:35.53983
[10]	validation_0-rmse:35.12292
[11]	validation_0-rmse:34.86331
[12]	validation_0-rmse:34.67659
[13]	validation_0-rmse:34.62496
[14]	validation_0-rmse:33.96615
[15]	validation_0-rmse:33.91632
[16]	validation_0-rmse:33.72175
[17]	validation_0-rmse:33.64442
[18]	validation_0-rmse:33.52384
[19]	validation_0-rmse:33.33246
[20]	validation_0-rmse:33.26280
[21]	validation_0-rmse:33.32197
[22]	validation_0-rmse:33.23083
[23]	validation_0-rmse:33.19608
[24]	validation_0-rmse:33.26166
[25]	validation_0-rmse:33.17399
[26]	validation_0-rmse:33.11543
[27]	validation_0-rmse:33.11131
[28]	validation_0-rmse:32.98413
[29]	validation_0-rmse:32.93300
[30]	validation_0-rmse:32.97844
[31]	validation_0-

In [12]:
results

{'Total Alkalinity':                         feature name    t-value  stat.significance  \
 0                   skin_temperature  18.392507       2.369725e-67   
 1                 cec_pH_interaction  14.926905       1.246000e-46   
 2                          elevation  11.401734       9.020171e-29   
 3                  flow_accumulation   8.811196       3.882902e-18   
 4                                EVI   8.675614       1.207699e-17   
 5                             swir22   6.425324       1.842704e-10   
 6    evaporation_precipitation_ratio   6.357067       2.841427e-10   
 7                               clay   4.749550       2.265404e-06   
 8           Land Surface Temperature   3.039231       2.419335e-03   
 9                                 pH   2.424059       1.548454e-02   
 10                  soil_temperature   2.322332       2.036946e-02   
 11        phosphorous_pH_interaction   2.189053       2.877116e-02   
 12  flow_acc_phosphorous_interaction   1.859463       6.

In [8]:
alk_feats = [
    "skin_temperature",
    "cec_pH_interaction",
    "elevation",
    "flow_accumulation",
    "EVI",
    "swir22",
    "evaporation_precipitation_ratio",
    "clay",
    "Land Surface Temperature",
    "pH",
    "soil_temperature",
    "phosphorous_pH_interaction"
]

len(alk_feats)

12

In [9]:
elec_feats = [
    "pet",
    "NDVI_LST_interaction",
    "flow_acc_phosphorous_interaction",
    "evaporation_precipitation_ratio",
    "phosphorous_pH_interaction",
    "soil_temperature",
    "clay",
    "total_evaporation_sum",
    "phosphorous",
    "Land Surface Temperature",
    "elevation",
    "cec_pH_interaction"
]

len(elec_feats)

12

In [10]:
drp_feats = [
    "phosphorous_pH_interaction",
    "soil_temperature",
    "clay",
    "pet",
    "total_evaporation_sum",
    "pH",
    "NDVI_LST_interaction",
    "NDMI",
    "cec_pH_interaction",
    "nir",
    "elevation",
    "green"
]

len(drp_feats)

12

In [11]:
selected_feats = list(set(alk_feats + elec_feats + drp_feats))

len(selected_feats)

20

## **Hyperparamter Optimization**

In [12]:
# Define new X_train and X_test based on feature selection results

X_train_fe = X_train[selected_feats]
X_test_fe = X_test[selected_feats]

X_train_fe.head()

,nir,total_evaporation_sum,EVI,phosphorous,pet,green,clay,pH,NDMI,cec_pH_interaction,elevation,NDVI_LST_interaction,skin_temperature,phosphorous_pH_interaction,evaporation_precipitation_ratio,soil_temperature,Land Surface Temperature,flow_accumulation,swir22,flow_acc_phosphorous_interaction
9297,18988.0,-0.000666,2142.0,19.0,218.20000,11249.5,25.0,66.0,0.050352,1584.0,1251.639625,62819712.0,303.094239,1254.0,-43.684083,303.376458,15936.0,1.000000,13577.0,19.000000
6378,13386.0,-0.000064,1247.0,21.0,225.60000,10901.0,27.0,73.0,-0.072333,1825.0,1012.728952,31571304.0,287.940814,1533.0,-34.438005,288.664629,14698.0,175.000000,13434.5,3675.000000
2420,11392.0,-0.000864,2681.0,26.0,210.20000,8999.0,24.0,70.0,0.058859,1540.0,171.655566,88964670.0,281.286717,1820.0,-0.532222,282.291864,14370.0,14123.536585,8997.0,367211.951220
6245,14316.0,-0.001014,1588.0,25.0,150.40001,10061.5,33.0,58.0,-0.061184,1276.0,1471.364299,44396740.0,276.171721,1450.0,-545.844320,276.449640,14585.0,65.000000,13487.5,1625.000000
4228,14446.5,-0.001255,1396.0,26.0,155.10000,10435.5,25.0,64.0,-0.078535,1536.0,898.500244,50134676.0,284.493027,1664.0,-9.226335,284.857243,14591.0,5514.232558,13974.0,143370.046512


In [19]:
# Next steps: perform hyperparameter optimization using Optuna. Then, using optimal
# hyperparameters retrain model for final testing.

def objective(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 100.0, log=True)
    solver = trial.suggest_categorical("solver", ["auto", "cholesky", "svd", "sag", "saga"])

    pipe = make_pipeline(preprocessor, MultiOutputRegressor(Ridge(alpha=alpha, solver=solver)))
    
    score = cross_val_score(pipe, X_train_fe, y_train, cv=5, scoring="r2", n_jobs=-1).mean()

    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=300)

[I 2026-03-12 17:36:17,230] A new study created in memory with name: no-name-a1a03038-c501-4b6b-8be9-db1a00a97650
[I 2026-03-12 17:36:18,811] Trial 0 finished with value: 0.26987427272831555 and parameters: {'alpha': 27.394312140538556, 'solver': 'cholesky'}. Best is trial 0 with value: 0.26987427272831555.
/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did n

In [20]:
study.best_params

{'alpha': 0.16103344304967634, 'solver': 'cholesky'}

In [21]:
study.best_value

0.27309957503712284

Clearly not performing very well, but for the purposes of stacking, lets continue to test it.

In [22]:
# Test performance with Regressor Chaining, given the optimized hyperparamters.
from itertools import permutations

targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
combinations = list(permutations(range(len(targets))))

# alkalinity = 0, electrical conductance = 1, dissolved reactive phosphorous = 2
combinations

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]

In [23]:
# Test with all possible chain combinations!

ridge = Ridge(alpha=0.16, solver='cholesky')
scores = []

for combo in combinations:
    chain = RegressorChain(base_estimator=ridge, order=combo)
    pipe = make_pipeline(preprocessor, chain)

    cv_scores = cross_validate(pipe, X_train_fe, y_train, cv=5, return_train_score=True, n_jobs=-1)

    scores.append({
        'order': combo,
        'mean_train_score': np.mean(cv_scores['train_score']),
        'mean_test_score': np.mean(cv_scores['test_score']),
        "mean_fit_time": np.mean(cv_scores["fit_time"]),
        "mean_score_time": np.mean(cv_scores["score_time"])
    })

In [25]:
pd.DataFrame(scores)

,order,mean_train_score,mean_test_score,mean_fit_time,mean_score_time
0,"(0, 1, 2)",0.278975,0.2731,0.022816,0.001588
1,"(0, 2, 1)",0.278975,0.2731,0.011703,0.001636
2,"(1, 0, 2)",0.278975,0.2731,0.012251,0.001361
3,"(1, 2, 0)",0.278975,0.2731,0.011741,0.001691
4,"(2, 0, 1)",0.278975,0.2731,0.011292,0.001268
5,"(2, 1, 0)",0.278975,0.2731,0.013060,0.001815


In [26]:
## FINAL MODEL EVALUATION ON TEST SET TO SEE PERFORMANCE, using Multi Output Regressor

final_ridge = make_pipeline(preprocessor, MultiOutputRegressor(Ridge(alpha=0.16, solver='cholesky')))

final_ridge.fit(X_train_fe, y_train)
score = final_ridge.score(X_test_fe, y_test)

score

0.2726392670090901

In [27]:
## Predict Output for Submission Set

submission_df = pd.read_csv('../data/validation_set.csv')
submission_df.head()

,Unnamed: 0,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN,161.90001,780.299056,1628.0,...,0.000007,0.136892,0.000000,1742.0,1742.0,4.542096e+07,351115.714286,338111.428571,-95.760822,0.962963
1,1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,177.60000,279.351413,2933.0,...,0.000550,0.259777,27.521283,1495.0,1690.0,1.069767e+08,202225.000000,210314.000000,-3.653135,0.920000
2,2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN,158.40001,163.622679,4032.0,...,0.000174,0.232120,14.464016,1403.0,1525.0,1.034820e+08,121744.000000,126816.666667,-10.921542,0.958333
3,3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,130.00000,44.046365,4701.0,...,NaN,NaN,42.085633,1121.0,1416.0,1.123978e+08,18889.531915,18889.531915,NaN,0.791667
4,4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN,152.50000,333.249422,1498.0,...,0.000618,0.336999,30.467072,1674.0,1674.0,6.468512e+07,410099.609756,381816.878049,-4.473473,0.931034


In [28]:
check_df = submission_df.drop(columns=['Unnamed: 0', 'Latitude', 
'Longitude', 'Sample Date', 
"Total Alkalinity", "Electrical Conductance",
'Dissolved Reactive Phosphorus'])

check_df.head()

,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,swir16,swir22,NDMI,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,161.90001,780.299056,1628.0,3016.0,15060.0,15229.0,12868.0,14797.0,12421.0,0.014388,...,0.000007,0.136892,0.000000,1742.0,1742.0,4.542096e+07,351115.714286,338111.428571,-95.760822,0.962963
1,177.60000,279.351413,2933.0,7263.0,14729.0,NaN,NaN,NaN,NaN,NaN,...,0.000550,0.259777,27.521283,1495.0,1690.0,1.069767e+08,202225.000000,210314.000000,-3.653135,0.920000
2,158.40001,163.622679,4032.0,7036.0,14707.5,16221.0,9304.5,12536.5,9958.0,0.128123,...,0.000174,0.232120,14.464016,1403.0,1525.0,1.034820e+08,121744.000000,126816.666667,-10.921542,0.958333
3,130.00000,44.046365,4701.0,7474.5,15037.5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,42.085633,1121.0,1416.0,1.123978e+08,18889.531915,18889.531915,NaN,0.791667
4,152.50000,333.249422,1498.0,4257.0,15195.0,9125.0,11100.5,9455.0,8711.0,-0.017761,...,0.000618,0.336999,30.467072,1674.0,1674.0,6.468512e+07,410099.609756,381816.878049,-4.473473,0.931034


In [29]:
test_df = check_df[selected_feats]

test_df.head()

,nir,total_evaporation_sum,EVI,phosphorous,pet,green,clay,pH,NDMI,cec_pH_interaction,elevation,NDVI_LST_interaction,skin_temperature,phosphorous_pH_interaction,evaporation_precipitation_ratio,soil_temperature,Land Surface Temperature,flow_accumulation,swir22,flow_acc_phosphorous_interaction
0,15229.0,-0.000788,1628.0,26.0,161.90001,12868.0,27.0,67.0,0.014388,1742.0,780.299056,4.542096e+07,288.386807,1742.0,-95.760822,288.264334,15060.0,13004.285714,12421.0,338111.428571
1,NaN,-0.002014,2933.0,26.0,177.60000,NaN,25.0,65.0,NaN,1495.0,279.351413,1.069767e+08,288.667907,1690.0,-3.653135,289.132705,14729.0,8089.000000,NaN,210314.000000
2,16221.0,-0.001915,4032.0,25.0,158.40001,9304.5,24.0,61.0,0.128123,1403.0,163.622679,1.034820e+08,291.107708,1525.0,-10.921542,291.689069,14707.5,5072.666667,9958.0,126816.666667
3,NaN,NaN,4701.0,24.0,130.00000,NaN,24.0,59.0,NaN,1121.0,44.046365,1.123978e+08,NaN,1416.0,NaN,NaN,15037.5,787.063830,NaN,18889.531915
4,9125.0,-0.002771,1498.0,27.0,152.50000,11100.5,29.0,62.0,-0.017761,1674.0,333.249422,6.468512e+07,288.706654,1674.0,-4.473473,289.314471,15195.0,14141.365854,8711.0,381816.878049


In [30]:
vals = final_ridge.predict(test_df)
val_df = pd.DataFrame(vals)
val_df.columns = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

val_df

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,117.791858,340.964798,14.293684
1,82.681887,437.774868,28.408598
2,74.735185,420.600599,27.648816
3,52.519699,397.892213,18.799831
4,66.903507,235.008298,11.388053
...,...,...,...
195,63.247680,283.789155,4.623760
196,45.808492,286.341846,10.967656
197,93.860789,245.483984,20.972063
198,89.693633,269.390241,18.001099


In [31]:
submission = pd.DataFrame({
    'Longitude': submission_df['Longitude'],
    'Latitude': submission_df['Latitude'],
    'Sample Date': submission_df['Sample Date'],
    'Total Alkalinity': val_df['Total Alkalinity'],
    'Electrical Conductance': val_df['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': val_df['Dissolved Reactive Phosphorus']
})

submission

,Longitude,Latitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,01-09-2014,117.791858,340.964798,14.293684
1,26.077500,-33.329167,16-09-2015,82.681887,437.774868,28.408598
2,27.640028,-32.991639,07-05-2015,74.735185,420.600599,27.648816
3,24.439167,-34.096389,07-02-2012,52.519699,397.892213,18.799831
4,28.581667,-32.000556,01-10-2014,66.903507,235.008298,11.388053
...,...,...,...,...,...,...
195,25.386667,-33.771111,06-12-2012,63.247680,283.789155,4.623760
196,27.390750,-33.185361,04-09-2014,45.808492,286.341846,10.967656
197,27.822778,-32.043333,28-09-2015,93.860789,245.483984,20.972063
198,25.161389,-33.001667,08-01-2015,89.693633,269.390241,18.001099


In [32]:
submission.to_csv('../submission.csv', index=False)